# Pamoka 03 - Agentiniai dizaino modeliai

Šioje pamokoje nagrinėjame tris kertinius dizaino modelius, skirtus efektyviems DI agentams kurti:

1. **Aiškios agento instrukcijos** — Tikslūs, vaidmenį apibrėžiantys užklausimai, kurie nukreipia agento elgesį
2. **Struktūruotas išvestis su Pydantic modeliais** — Užtikrinimas, kad agentai grąžintų prognozuojamus, patikrintus duomenis
3. **Vienos atsakomybės agentai** — Fokusuotų agentų projektavimas, kurie gerai atlieka po vieną užduotį

Kiekvieną modelį pritaikysime **kelionių krypčių rekomendatoriui**, palaipsniui kuriant sistemą, galinčią pasiūlyti krypčių, patikrinti jų prieinamumą ir tvarkyti logistiką.


## Sąranka


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic python-dotenv --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## 1 modelis: aiškios agento instrukcijos

Sakytume, kad pati įtakingiausia ir paprasčiausia schema yra rašyti aiškias, detalias instrukcijas savo agentui.

Geros instrukcijos apibrėžia:
- **Kas** yra agentas (asmenybė ir tonas)
- **Ką** jis turėtų daryti (užduotys žingsnis po žingsnio)
- **Kaip** jis turėtų elgtis (ribojimai ir stilius)

Žemiau sukuriame kelionių konsjeržo agentą su aiškiomis instrukcijomis, kurios formuoja kiekvieną jo atsakymą.


In [ ]:
agent = provider.as_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## Šablonas 2: Struktūruotas išvestis su Pydantic modeliais

Laisvos formos tekstas yra naudingas pokalbiams, tačiau tolesnės sistemos reikalauja struktūruotų duomenų. 
Derindami **Pydantic modelius** su **įrankio funkcija**, galime:

- Apibrėžti tikslų agento išvesties schemą
- Automatiškai tikrinti atsakymus
- Patikimai integruoti agento rezultatus į programos logiką

Taip pat pristatome įrankį, kuris grąžina paskirties vietos duomenis, kad agentas pagrįstų savo rekomendacijas tikrais duomenimis.


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(destination: Annotated[str, "The destination to look up"]) -> str:
    """Get details about a vacation destination."""
    details = {
        "Barcelona": "Available. Best: May-Jun. Beach, architecture, nightlife. ~$2000/week",
        "Tokyo": "Available. Best: Mar-Apr. Culture, food, technology. ~$2500/week",
        "Cape Town": "Not available. Best: Nov-Mar. Nature, wine, adventure. ~$1800/week",
    }
    return details.get(destination, f"{destination}: No information available.")


structured_agent = provider.as_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget"
)

if response:
    print(response)

## Šablonas 3: Vienos atsakomybės agentai

Sudėtingos užduotys naudingos, kai darbas yra padalintas tarp kelių specializuotų agentų, kiekvienas turintis vieną atsakomybę:

- **Vietos ekspertas**, kuris žino apie vietas ir jų prieinamumą
- **Logistikos planuotojas**, kuris rūpinasi skrydžiais, viešbučiais ir maršrutais

Tai atspindi programinės įrangos inžinerijos principą *atskyrimo rūpesčiais* — kiekvieną agentą lengviau testuoti, prižiūrėti ir tobulinti nepriklausomai.


In [ ]:
destination_agent = provider.as_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = provider.as_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## Santrauka

Šioje pamokoje pritaikėme tris agentinio dizaino šablonus kelionių rekomendacijų scenarijui:

| Šablonas | Pagrindinė idėja | Nauda |
|---|---|---|
| **Aiškios instrukcijos** | Iš anksto apibrėžkite personą, atsakomybes ir apribojimus | Nuoseklus, į prekės ženklą orientuotas agento elgesys |
| **Struktūruotas išvestis** | Naudokite Pydantic modelius kaip atsakymo formatą | Patikrinti, mašinai skaitomi rezultatai |
| **Viena atsakomybė** | Kiekvienam agentui skirkite vieną sritį | Lengviau testuoti, prižiūrėti ir derinti |

Šie šablonai natūraliai jungiasi — galite derinti aiškias instrukcijas su struktūruotu išvestimi viename atsakomybės agento viduje, kad sukurtumėte patikimas, gamybai paruoštas sistemas.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Atsakomybės apribojimas**:
Šis dokumentas buvo išverstas naudojant dirbtinio intelekto vertimo paslaugą [Co-op Translator](https://github.com/Azure/co-op-translator). Nors siekiame tikslumo, prašome atkreipti dėmesį, kad automatiniai vertimai gali turėti klaidų ar netikslumų. Originalus dokumentas jo gimtąja kalba laikomas autoritetingu šaltiniu. Svarbiai informacijai rekomenduojama naudoti profesionalų žmogiškąjį vertimą. Mes neatsakome už jokius nesusipratimus ar neteisingą interpretaciją, kilusią naudojantis šiuo vertimu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
